In [15]:
import wandb
wandb.login()
api = wandb.Api()

run_path="tidiane/patch_icl/qih3cd3k"

In [16]:
# Get run metrics from wandb
run = api.run(run_path)
metrics = run.history()
metrics.head()

,train_batch/context_aggreg_loss,train_batch/final_soft_dice,train_batch/local_dice,train_batch/level_2_target_combined_loss,train_batch/level_1_alpha,train_batch/level_2_target_aggreg_loss,train_batch/target_loss,train_batch/final_pixel_mae,train_batch/context_dice,train_batch/level_0_conf_supervision_loss,...,val_dice/iliopsoas_right,val_dice/sacrum,train_dice/rib_left_4,train_dice/clavicula_right,val/final,train_batch/level_2_refined_probs_soft_dice,train_batch/level_1_refined_probs_soft_dice,train_batch/level_1_refined_probs_dice,train_batch/level_2_refined_probs_dice,train/by_level
0,0.301402,0.389653,0.265677,0.624802,0.995493,0.426203,1.359642,0.019377,0.498072,0.012850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.320129,0.401961,0.279803,0.595073,0.995249,0.432560,1.319030,0.017335,0.486138,0.011306,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.323643,0.399180,0.285880,0.598970,0.995202,0.437106,1.323666,0.019344,0.474403,0.011237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.323525,0.398125,0.284087,0.600182,0.995198,0.437728,1.325107,0.019186,0.475496,0.011213,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.322461,0.398702,0.284231,0.599236,0.995184,0.436942,1.323475,0.019047,0.475760,0.011219,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Count rows with valid 'val/final_dice' values
valid_dice_count = metrics['val/final_dice'].notna().sum()
print(f"Rows with valid 'val/final_dice': {valid_dice_count}")

Rows with valid 'val/final_dice': 3


In [ ]:
# Fetch only rows where 'val/final_dice' was logged (avoids sampling issues)
val_metrics = run.history(keys=['val/final_dice'])
print(val_metrics.to_markdown(index=False))

|   train_batch/level_2_target_patch_loss |   train_batch/patch_loss_total |   train_batch/level_2_dice |   train_batch/mean_confidence |   train_batch/level_0_dice |   train_batch/level_1_oracle_prob |   train_batch/level_2_target_combined_loss |   _step |   train_batch/level_1_alpha |   train_batch/level_0_soft_dice |   train_batch/final_soft_dice |   train_batch/level_2_soft_dice |   train_batch/local_dice |   global_step |   train_batch/total_loss |   train_batch/level_1_conf_supervision_loss |   train_batch/level_1_dice |   train_batch/level_0_conf_supervision_loss |   _runtime |   train_batch/level_0_context_patch_loss |   _timestamp |   train_batch/level_1_soft_dice |   train_batch/level_1_target_aggreg_loss |   train_batch/final_pixel_mae |   train_batch/aggreg_loss_total |   train_batch/level_2_context_aggreg_loss |   train_batch/mean_entropy |   train_batch/level_1_target_patch_loss |   train_batch/context_aggreg_loss |   train_batch/level_2_pixel_mae |   train_batch/level_2_

In [14]:
filtered_metrics["step"]

KeyError: 'step'

In [19]:
# Diagnose the sampling issue
print(f"Total rows from run.history() (default sampling ~500): {len(metrics)}")
print(f"Columns with 'val': {[c for c in metrics.columns if 'val' in c.lower()]}")
print(f"'_step' range: {metrics['_step'].min()} to {metrics['_step'].max()}")
print(f"Non-NA val/final_dice count: {metrics['val/final_dice'].notna().sum()}")
print()

# The issue: run.history() defaults to ~500 sampled rows.
# Training metrics are logged every step, but validation metrics only every N steps (e.g. once per epoch).
# When wandb samples 500 points from thousands of steps, most sampled rows won't coincide
# with a validation step, so val columns are NaN.

# Fix: use run.history(keys=...) to only fetch rows where the key was logged
val_metrics = run.history(keys=['val/final_dice'])
print(f"Rows with val/final_dice using keys filter: {len(val_metrics)}")
print(val_metrics[['_step', 'val/final_dice']])

Total rows from run.history(): 500
Columns with 'val' in name: ['val/by_level', 'val_dice/vertebrae_T11', 'val_dice/rib_left_7', 'val_dice/rib_right_9', 'val/target_mask_size_min', 'val_dice/colon', 'val/context_mask_size_mean', 'val_local_softdice', 'val_dice/rib_right_7', 'val_dice/rib_left_12', 'val_dice/vertebrae_T9', 'val_dice/vertebrae_L4', 'val/local_soft_dice', 'val_dice/iliopsoas_left', 'val/target_mask_size_mean', 'val/final_pixel_mae', 'val/level_0_pixel_mae', 'val_context_softdice', 'val/level_1_soft_dice', 'val_loss', 'val/level_2_dice', 'val_dice/kidney_left', 'val_dice/rib_right_12', 'val_dice/costal_cartilages', 'val_dice/atrial_appendage_left', 'val_dice/iliac_artery_left', 'val_dice/rib_right_8', 'val/level_2_soft_dice', 'val/target_mask_size_max', 'val_dice/kidney_right', 'val/per_case', 'val_dice/rib_left_8', 'val/mean_entropy_on_correct', 'val/context_soft_dice', 'val/level_1_refined_probs_soft_dice', 'val_dice/liver', 'val/level_0_soft_dice', 'val_dice/vertebrae_T

In [20]:
# Find all validation-related keys in the run
val_keys = [k for k in run.summary.keys() if 'val' in k.lower()]
print("Validation keys:", val_keys)

Validation keys: ['val/by_level', 'val/context_dice', 'val/context_mask_size_max', 'val/context_mask_size_mean', 'val/context_mask_size_min', 'val/context_soft_dice', 'val/final', 'val/final_dice', 'val/final_pixel_mae', 'val/final_soft_dice', 'val/level_0_dice', 'val/level_0_pixel_mae', 'val/level_0_soft_dice', 'val/level_1_dice', 'val/level_1_pixel_mae', 'val/level_1_refined_probs_dice', 'val/level_1_refined_probs_soft_dice', 'val/level_1_soft_dice', 'val/level_2_dice', 'val/level_2_pixel_mae', 'val/level_2_refined_probs_dice', 'val/level_2_refined_probs_soft_dice', 'val/level_2_soft_dice', 'val/local_dice', 'val/local_soft_dice', 'val/mean_confidence', 'val/mean_entropy', 'val/mean_entropy_on_correct', 'val/mean_entropy_on_errors', 'val/per_case', 'val/target_mask_size_max', 'val/target_mask_size_mean', 'val/target_mask_size_min', 'val_context_dice', 'val_context_softdice', 'val_dice/atrial_appendage_left', 'val_dice/autochthon_left', 'val_dice/autochthon_right', 'val_dice/brachioce

In [25]:
# Fetch all epoch-end rows with all val/ columns
import pandas as pd

# Use val/final_dice as anchor to get epoch-end steps, then fetch all val/ keys
epoch_end_metrics = run.history(keys=['val/final_dice'])
val_slash_keys = [k for k in val_keys if k.startswith('val/')]

# Fetch each val/ key separately and merge on _step
for key in val_slash_keys:
    if key == 'val/final_dice':
        continue
    key_df = run.history(keys=[key])
    if len(key_df) > 0:
        epoch_end_metrics = epoch_end_metrics.merge(key_df[['_step', key]], on='_step', how='left')

# Remove columns with any missing values
epoch_end_metrics = epoch_end_metrics.dropna(axis=1)

print(f"Total epoch-end rows: {len(epoch_end_metrics)}")
print(f"Columns: {list(epoch_end_metrics.columns)}")
epoch_end_metrics

Total epoch-end rows: 17
Columns: ['_step', 'val/final_dice', 'val/context_dice', 'val/context_mask_size_max', 'val/context_mask_size_mean', 'val/context_mask_size_min', 'val/context_soft_dice', 'val/final_pixel_mae', 'val/final_soft_dice', 'val/level_0_dice', 'val/level_0_pixel_mae', 'val/level_0_soft_dice', 'val/level_1_dice', 'val/level_1_pixel_mae', 'val/level_1_refined_probs_dice', 'val/level_1_refined_probs_soft_dice', 'val/level_1_soft_dice', 'val/level_2_dice', 'val/level_2_pixel_mae', 'val/level_2_refined_probs_dice', 'val/level_2_refined_probs_soft_dice', 'val/level_2_soft_dice', 'val/local_dice', 'val/local_soft_dice', 'val/mean_confidence', 'val/mean_entropy', 'val/mean_entropy_on_correct', 'val/mean_entropy_on_errors', 'val/per_case', 'val/target_mask_size_max', 'val/target_mask_size_mean', 'val/target_mask_size_min']


,_step,val/final_dice,val/context_dice,val/context_mask_size_max,val/context_mask_size_mean,val/context_mask_size_min,val/context_soft_dice,val/final_pixel_mae,val/final_soft_dice,val/level_0_dice,...,val/local_dice,val/local_soft_dice,val/mean_confidence,val/mean_entropy,val/mean_entropy_on_correct,val/mean_entropy_on_errors,val/per_case,val/target_mask_size_max,val/target_mask_size_mean,val/target_mask_size_min
0,5768,0.331679,0.572664,6395,101.488298,0,0.648917,0.011794,0.341378,0.518076,...,0.161515,0.171692,0.986484,0.013515,0.008111,0.469007,{'sha256': '33cac0e33161d0ec18ab352ad8dd2d51cb...,6395,101.243214,0
1,7691,0.330107,0.572671,6395,101.475217,0,0.648919,0.011704,0.340999,0.518172,...,0.161421,0.172055,0.987218,0.012782,0.007555,0.454991,{'_latest_artifact_path': 'wandb-client-artifa...,6395,101.243214,0
2,9614,0.326398,0.572223,6395,101.457539,0,0.648360,0.012161,0.337352,0.518670,...,0.159669,0.170244,0.986566,0.013434,0.007935,0.460028,{'sha256': '77f99dd1e04f9f3940821a9165d27b3651...,6395,101.243214,0
3,11537,0.322768,0.572584,6395,101.433388,0,0.649064,0.012410,0.334888,0.520840,...,0.157849,0.168856,0.986555,0.013445,0.007935,0.451159,"{'ncols': 6, 'size': 760311, '_type': 'table-f...",6395,101.243214,0
4,13460,0.324333,0.572381,6395,101.409955,0,0.648641,0.012358,0.336383,0.520352,...,0.158228,0.169287,0.986515,0.013485,0.007962,0.454256,{'path': 'media/table/val/per_case_13460_d4a65...,6395,101.243214,0
5,15383,0.322931,0.571919,6395,101.453169,0,0.648401,0.012608,0.336267,0.521530,...,0.156869,0.168349,0.986655,0.013345,0.007745,0.448046,"{'_type': 'table-file', 'artifact_path': 'wand...",6395,101.243214,0
6,17306,0.321528,0.572207,6395,101.445517,0,0.648660,0.012788,0.334607,0.523220,...,0.155131,0.166525,0.986340,0.013660,0.007992,0.448366,{'_latest_artifact_path': 'wandb-client-artifa...,6395,101.243214,0
7,19229,0.332259,0.571962,6395,101.430993,0,0.648594,0.011935,0.344394,0.523661,...,0.160490,0.171634,0.987620,0.012380,0.007201,0.439301,"{'nrows': 28148, 'log_mode': 'IMMUTABLE', 'sha...",6395,101.243214,0
8,21152,0.296392,0.571199,6395,101.452849,0,0.648549,0.014155,0.314916,0.507671,...,0.141918,0.155575,0.985230,0.014770,0.008274,0.457413,"{'_type': 'table-file', 'artifact_path': 'wand...",6395,101.243214,0
9,23075,0.342039,0.571686,6395,101.436138,0,0.648159,0.010815,0.349832,0.515449,...,0.162026,0.170226,0.987358,0.012642,0.007225,0.507176,{'sha256': '569d74650c239a918835dcc01af2dbac64...,6395,101.243214,0


In [28]:
# Remove val/per_case, print and save to CSV
import pandas as pd
epoch_end_metrics = epoch_end_metrics.drop(columns=['val/per_case'], errors='ignore')
print(epoch_end_metrics.to_markdown(index=False))
epoch_end_metrics.to_csv('epoch_end_metrics.csv', index=False)
print("\nSaved to epoch_end_metrics.csv")

|   _step |   val/final_dice |   val/context_dice |   val/context_mask_size_max |   val/context_mask_size_mean |   val/context_mask_size_min |   val/context_soft_dice |   val/final_pixel_mae |   val/final_soft_dice |   val/level_0_dice |   val/level_0_pixel_mae |   val/level_0_soft_dice |   val/level_1_dice |   val/level_1_pixel_mae |   val/level_1_refined_probs_dice |   val/level_1_refined_probs_soft_dice |   val/level_1_soft_dice |   val/level_2_dice |   val/level_2_pixel_mae |   val/level_2_refined_probs_dice |   val/level_2_refined_probs_soft_dice |   val/level_2_soft_dice |   val/local_dice |   val/local_soft_dice |   val/mean_confidence |   val/mean_entropy |   val/mean_entropy_on_correct |   val/mean_entropy_on_errors |   val/target_mask_size_max |   val/target_mask_size_mean |   val/target_mask_size_min |
|--------:|-----------------:|-------------------:|----------------------------:|-----------------------------:|----------------------------:|------------------------:|-------

In [29]:
import sys
sys.path.insert(0, "/home/dpxuser/ic_segmentation")

import pandas as pd
import numpy as np

# Load the metrics
df = pd.read_csv("/home/dpxuser/ic_segmentation/experiments/epoch_end_metrics.csv")
print(f"Shape: {df.shape}")
print(f"Epochs: {len(df)} (steps: {df['_step'].min()} to {df['_step'].max()})")
print(f"\nColumns ({len(df.columns)}):")
for col in sorted(df.columns):
    print(f"  {col}")


FileNotFoundError: [Errno 2] No such file or directory: '/home/dpxuser/ic_segmentation/experiments/epoch_end_metrics.csv'

In [30]:
import os

# List likely locations
paths_to_check = [
    "/home/dpxuser/ic_segmentation/experiments",
    "/home/dpxuser/ic_segmentation",
    "/home/dpxuser",
    "/workspaces",
    os.getcwd(),
]

for path in paths_to_check:
    print(f"\n{path}:")
    try:
        print(os.listdir(path)[:20])
    except Exception as e:
        print(f"  Error: {e}")



/home/dpxuser/ic_segmentation/experiments:
  Error: [Errno 2] No such file or directory: '/home/dpxuser/ic_segmentation/experiments'

/home/dpxuser/ic_segmentation:
  Error: [Errno 2] No such file or directory: '/home/dpxuser/ic_segmentation'

/home/dpxuser:
['matlab_crash_dump.43736-1', '.jupyter', '.keras', 'matlab_crash_dump.43853-1', 'matlab_crash_dump.44514-1', '.bashrc', '.conda', 'matlab_crash_dump.53221-1', '.config', '.java', '.singularity', '.ssh', 'matlab_crash_dump.44849-1', 'matlab_crash_dump.54515-1', 'nohup.out', '.wget-hsts', 'matlab_crash_dump.53778-1', '.Xauthority', 'matlab_crash_dump.44543-1', 'matlab_crash_dump.44599-1']

/workspaces:
  Error: [Errno 2] No such file or directory: '/workspaces'

/software/notebooks/camaret/ic_segmentation/experiments:
['eval_neuroverse3d.py', 'medsegbench_sample.png', 'feature_extraction', 'dataloading', '__init__.py', 'eval_medverse.py', 'epoch_end_metrics.csv', 'wandb_stats.ipynb']


In [ ]:
import pandas as pd
import numpy as np

# Use the correct path
df = pd.read_csv("/software/notebooks/camaret/ic_segmentation/experiments/epoch_end_metrics.csv")
print(f"Shape: {df.shape}")
print(f"Epochs: {len(df)} (steps: {df['_step'].min()} to {df['_step'].max()})")
print(f"\nColumns ({len(df.columns)}):")
for col in sorted(df.columns):
    print(f"  {col}")


In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv("/software/notebooks/camaret/ic_segmentation/experiments/epoch_end_metrics.csv")
print(f"Shape: {df.shape}, Epochs: {len(df)}")
print(f"\nColumns: {list(df.columns)}")


Shape: (17, 31), Epochs: 17

Columns: ['_step', 'val/final_dice', 'val/context_dice', 'val/context_mask_size_max', 'val/context_mask_size_mean', 'val/context_mask_size_min', 'val/context_soft_dice', 'val/final_pixel_mae', 'val/final_soft_dice', 'val/level_0_dice', 'val/level_0_pixel_mae', 'val/level_0_soft_dice', 'val/level_1_dice', 'val/level_1_pixel_mae', 'val/level_1_refined_probs_dice', 'val/level_1_refined_probs_soft_dice', 'val/level_1_soft_dice', 'val/level_2_dice', 'val/level_2_pixel_mae', 'val/level_2_refined_probs_dice', 'val/level_2_refined_probs_soft_dice', 'val/level_2_soft_dice', 'val/local_dice', 'val/local_soft_dice', 'val/mean_confidence', 'val/mean_entropy', 'val/mean_entropy_on_correct', 'val/mean_entropy_on_errors', 'val/target_mask_size_max', 'val/target_mask_size_mean', 'val/target_mask_size_min']


In [32]:
# Compute comprehensive statistics
print("=" * 70)
print("PIPELINE ARCHITECTURE SUMMARY (config 110_learned_confidence)")
print("=" * 70)
print("""
3-Level Cascaded PatchICL:
  Level 0: 20px  (8x8 patches, stride 7, 9 target + 4 context patches)
  Level 1: 60px  (8x8 patches, stride 6, 12 target + 4 context patches)  
  Level 2: 128px (8x8 patches, stride 6, 20 target + 4 context patches)

Key Features:
  - Learned confidence prediction (predict_confidence: true)
  - Entropy-based sampling at high-uncertainty regions
  - Oracle scheduling: warmup 10 epochs, decay 30 epochs, end at 0.2
  - Learned alpha blending between levels (from register tokens)
  - Confidence-weighted aggregation with detached gradients
""")

print("=" * 70)
print("VALIDATION METRICS STATISTICS (17 epochs)")
print("=" * 70)

# Core segmentation metrics
core_metrics = ['val/final_dice', 'val/final_soft_dice', 'val/final_pixel_mae',
                'val/level_0_dice', 'val/level_1_dice', 'val/level_2_dice',
                'val/level_0_soft_dice', 'val/level_1_soft_dice', 'val/level_2_soft_dice']

print("\n--- DICE SCORES BY LEVEL ---")
for col in ['val/level_0_dice', 'val/level_1_dice', 'val/level_2_dice', 'val/final_dice']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:30s} | mean: {data.mean():.4f} | std: {data.std():.4f} | min: {data.min():.4f} | max: {data.max():.4f}")

print("\n--- SOFT DICE SCORES BY LEVEL ---")
for col in ['val/level_0_soft_dice', 'val/level_1_soft_dice', 'val/level_2_soft_dice', 'val/final_soft_dice']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:30s} | mean: {data.mean():.4f} | std: {data.std():.4f} | min: {data.min():.4f} | max: {data.max():.4f}")


PIPELINE ARCHITECTURE SUMMARY (config 110_learned_confidence)

3-Level Cascaded PatchICL:
  Level 0: 20px  (8x8 patches, stride 7, 9 target + 4 context patches)
  Level 1: 60px  (8x8 patches, stride 6, 12 target + 4 context patches)  
  Level 2: 128px (8x8 patches, stride 6, 20 target + 4 context patches)

Key Features:
  - Learned confidence prediction (predict_confidence: true)
  - Entropy-based sampling at high-uncertainty regions
  - Oracle scheduling: warmup 10 epochs, decay 30 epochs, end at 0.2
  - Learned alpha blending between levels (from register tokens)
  - Confidence-weighted aggregation with detached gradients

VALIDATION METRICS STATISTICS (17 epochs)

--- DICE SCORES BY LEVEL ---
level_0_dice                   | mean: 0.5222 | std: 0.0069 | min: 0.5077 | max: 0.5350
level_1_dice                   | mean: 0.4540 | std: 0.0242 | min: 0.4122 | max: 0.4957
level_2_dice                   | mean: 0.3259 | std: 0.0355 | min: 0.2663 | max: 0.3877
final_dice                     

In [33]:
print("\n--- REFINED PROBS DICE (sampling guidance quality) ---")
for col in ['val/level_1_refined_probs_dice', 'val/level_1_refined_probs_soft_dice',
            'val/level_2_refined_probs_dice', 'val/level_2_refined_probs_soft_dice']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:35s} | mean: {data.mean():.4f} | std: {data.std():.4f} | max: {data.max():.4f}")

print("\n--- CONFIDENCE & ENTROPY METRICS ---")
for col in ['val/mean_confidence', 'val/mean_entropy', 'val/mean_entropy_on_correct', 'val/mean_entropy_on_errors']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:30s} | mean: {data.mean():.4f} | std: {data.std():.4f} | min: {data.min():.4f} | max: {data.max():.4f}")

print("\n--- PIXEL MAE BY LEVEL ---")
for col in ['val/level_0_pixel_mae', 'val/level_1_pixel_mae', 'val/level_2_pixel_mae', 'val/final_pixel_mae']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:30s} | mean: {data.mean():.4f} | std: {data.std():.4f}")

print("\n--- CONTEXT METRICS ---")
for col in ['val/context_dice', 'val/context_soft_dice', 'val/local_dice', 'val/local_soft_dice']:
    data = df[col]
    name = col.replace('val/', '')
    print(f"{name:30s} | mean: {data.mean():.4f} | std: {data.std():.4f}")



--- REFINED PROBS DICE (sampling guidance quality) ---
level_1_refined_probs_dice          | mean: 0.1335 | std: 0.0269 | max: 0.1863
level_1_refined_probs_soft_dice     | mean: 0.1900 | std: 0.0077 | max: 0.2020
level_2_refined_probs_dice          | mean: 0.0230 | std: 0.0227 | max: 0.0596
level_2_refined_probs_soft_dice     | mean: 0.1283 | std: 0.0195 | max: 0.1620

--- CONFIDENCE & ENTROPY METRICS ---
mean_confidence                | mean: 0.9870 | std: 0.0013 | min: 0.9852 | max: 0.9915
mean_entropy                   | mean: 0.0130 | std: 0.0013 | min: 0.0085 | max: 0.0148
mean_entropy_on_correct        | mean: 0.0077 | std: 0.0008 | min: 0.0050 | max: 0.0083
mean_entropy_on_errors         | mean: 0.4968 | std: 0.0563 | min: 0.4393 | max: 0.5842

--- PIXEL MAE BY LEVEL ---
level_0_pixel_mae              | mean: 0.0182 | std: 0.0007
level_1_pixel_mae              | mean: 0.0126 | std: 0.0014
level_2_pixel_mae              | mean: 0.0107 | std: 0.0014
final_pixel_mae               

In [34]:
# Training progression analysis
print("\n" + "=" * 70)
print("TRAINING PROGRESSION ANALYSIS")
print("=" * 70)

# Compare first half vs second half
mid = len(df) // 2
first_half = df.iloc[:mid]
second_half = df.iloc[mid:]

print("\n--- DICE IMPROVEMENT (first half → second half) ---")
for col in ['val/level_0_dice', 'val/level_1_dice', 'val/level_2_dice', 'val/final_dice']:
    name = col.replace('val/', '')
    first = first_half[col].mean()
    second = second_half[col].mean()
    diff = second - first
    print(f"{name:20s} | {first:.4f} → {second:.4f} | Δ = {diff:+.4f}")

print("\n--- BEST EPOCH ANALYSIS ---")
best_final_idx = df['val/final_dice'].idxmax()
best_row = df.loc[best_final_idx]
print(f"Best final_dice at epoch index {best_final_idx} (step {best_row['_step']:.0f}):")
print(f"  final_dice:    {best_row['val/final_dice']:.4f}")
print(f"  level_0_dice:  {best_row['val/level_0_dice']:.4f}")
print(f"  level_1_dice:  {best_row['val/level_1_dice']:.4f}")
print(f"  level_2_dice:  {best_row['val/level_2_dice']:.4f}")
print(f"  confidence:    {best_row['val/mean_confidence']:.4f}")

# Last 3 epochs average (most recent performance)
print("\n--- LAST 3 EPOCHS AVERAGE ---")
last_3 = df.tail(3)
for col in ['val/final_dice', 'val/level_0_dice', 'val/level_1_dice', 'val/level_2_dice', 'val/mean_confidence']:
    name = col.replace('val/', '')
    print(f"{name:20s} | {last_3[col].mean():.4f}")



TRAINING PROGRESSION ANALYSIS

--- DICE IMPROVEMENT (first half → second half) ---
level_0_dice         | 0.5206 → 0.5237 | Δ = +0.0031
level_1_dice         | 0.4383 → 0.4679 | Δ = +0.0296
level_2_dice         | 0.3021 → 0.3470 | Δ = +0.0450
final_dice           | 0.3265 → 0.3759 | Δ = +0.0494

--- BEST EPOCH ANALYSIS ---
Best final_dice at epoch index 15 (step 34613):
  final_dice:    0.4176
  level_0_dice:  0.5305
  level_1_dice:  0.4957
  level_2_dice:  0.3877
  confidence:    0.9877

--- LAST 3 EPOCHS AVERAGE ---
final_dice           | 0.4075
level_0_dice         | 0.5317
level_1_dice         | 0.4906
level_2_dice         | 0.3765
mean_confidence      | 0.9874


In [35]:
print("\n" + "=" * 70)
print("KEY OBSERVATIONS & ANALYSIS")
print("=" * 70)

print("""
1. LEVEL PERFORMANCE HIERARCHY (inverted - coarser is better):
   Level 0 (20px):  0.522 dice  ← BEST individual level
   Level 1 (60px):  0.454 dice
   Level 2 (128px): 0.326 dice  ← WORST individual level
   Final (combined): 0.353 dice

   ISSUE: Fine-grained levels HURT performance. Final dice (0.353) is worse 
   than Level 0 alone (0.522). The cascade combination is not additive.

2. REFINED PROBS QUALITY (sampling guidance from model predictions):
   Level 1 refined_probs_dice: 0.133 (very poor - GT has ~0.5 dice at this res)
   Level 2 refined_probs_dice: 0.023 (near-random!)
   
   ISSUE: Model predictions at level N don't guide sampling well at level N+1.
   The entropy-based sampling from model uncertainty isn't working as intended.

3. CONFIDENCE CALIBRATION:
   Mean confidence:           0.987 (very high)
   Entropy on correct pixels: 0.008 (low - good)
   Entropy on error pixels:   0.497 (high - model knows when it's wrong)
   
   GOOD: Entropy is ~62x higher on errors vs correct. Model is somewhat calibrated.
   ISSUE: Overall confidence (98.7%) is too high given actual dice of 35%.

4. CONTEXT VS TARGET:
   Context dice:  0.573 (context patches perform well)
   Local dice:    0.163 (standalone patches without context are poor)
   
   GOOD: Context is providing significant benefit (+0.41 dice improvement)

5. TRAINING DYNAMICS:
   - Level 0 is stable (std 0.007)
   - Level 2 improves most (+0.045 dice first→second half)
   - Final dice tracks level 2 closely
""")

# Correlation analysis
print("\n--- CORRELATION: Level Dices vs Final Dice ---")
for col in ['val/level_0_dice', 'val/level_1_dice', 'val/level_2_dice']:
    corr = df[col].corr(df['val/final_dice'])
    name = col.replace('val/', '')
    print(f"{name:20s} corr with final_dice: {corr:.3f}")



KEY OBSERVATIONS & ANALYSIS

1. LEVEL PERFORMANCE HIERARCHY (inverted - coarser is better):
   Level 0 (20px):  0.522 dice  ← BEST individual level
   Level 1 (60px):  0.454 dice
   Level 2 (128px): 0.326 dice  ← WORST individual level
   Final (combined): 0.353 dice

   ISSUE: Fine-grained levels HURT performance. Final dice (0.353) is worse 
   than Level 0 alone (0.522). The cascade combination is not additive.

2. REFINED PROBS QUALITY (sampling guidance from model predictions):
   Level 1 refined_probs_dice: 0.133 (very poor - GT has ~0.5 dice at this res)
   Level 2 refined_probs_dice: 0.023 (near-random!)

   ISSUE: Model predictions at level N don't guide sampling well at level N+1.
   The entropy-based sampling from model uncertainty isn't working as intended.

3. CONFIDENCE CALIBRATION:
   Mean confidence:           0.987 (very high)
   Entropy on correct pixels: 0.008 (low - good)
   Entropy on error pixels:   0.497 (high - model knows when it's wrong)

   GOOD: Entropy is 

In [36]:
# Final summary table
print("\n" + "=" * 70)
print("SUMMARY STATISTICS TABLE")
print("=" * 70)

summary_cols = [
    ('val/final_dice', 'Final Dice'),
    ('val/final_soft_dice', 'Final Soft Dice'),
    ('val/level_0_dice', 'Level 0 Dice (20px)'),
    ('val/level_1_dice', 'Level 1 Dice (60px)'),
    ('val/level_2_dice', 'Level 2 Dice (128px)'),
    ('val/context_dice', 'Context Dice'),
    ('val/mean_confidence', 'Mean Confidence'),
    ('val/mean_entropy', 'Mean Entropy'),
]

print(f"\n{'Metric':<25} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Last':>8}")
print("-" * 70)
for col, name in summary_cols:
    d = df[col]
    print(f"{name:<25} {d.mean():>8.4f} {d.std():>8.4f} {d.min():>8.4f} {d.max():>8.4f} {d.iloc[-1]:>8.4f}")

print("\n" + "=" * 70)
print("ACTIONABLE INSIGHTS")
print("=" * 70)
print("""
1. CASCADE COMBINATION ISSUE:
   - Final dice (0.35) << Level 0 dice (0.52)
   - The learned_alpha blending is degrading coarse predictions
   - Consider: use level 0 only, or fix alpha initialization/training

2. SAMPLING GUIDANCE FAILURE:
   - refined_probs_dice << 0.1 means model predictions aren't guiding 
     patch selection effectively at finer levels
   - Consider: keep oracle_levels_train: [false, true, true] OR
     increase oracle_sched_end_prob to maintain GT guidance

3. OVERCONFIDENCE:
   - 98.7% confidence but only 35% dice suggests miscalibration
   - The confidence supervision loss may need higher weight
   - Consider: increase supervision_weight from 0.1 to 0.3+

4. PROMISING SIGNS:
   - Context provides +0.41 dice benefit (context dice 0.57 vs local 0.16)
   - Entropy discriminates errors (0.50) from correct (0.008) - 62x ratio
   - Training is still improving (final_dice: +0.05 first→second half)
""")



SUMMARY STATISTICS TABLE

Metric                        Mean      Std      Min      Max     Last
----------------------------------------------------------------------
Final Dice                  0.3526   0.0368   0.2964   0.4176   0.4027
Final Soft Dice             0.3594   0.0298   0.3149   0.4068   0.4058
Level 0 Dice (20px)         0.5222   0.0069   0.5077   0.5350   0.5297
Level 1 Dice (60px)         0.4540   0.0242   0.4122   0.4957   0.4909
Level 2 Dice (128px)        0.3259   0.0355   0.2663   0.3877   0.3697
Context Dice                0.5728   0.0009   0.5712   0.5748   0.5728
Mean Confidence             0.9870   0.0013   0.9852   0.9915   0.9868
Mean Entropy                0.0130   0.0013   0.0085   0.0148   0.0132

ACTIONABLE INSIGHTS

1. CASCADE COMBINATION ISSUE:
   - Final dice (0.35) << Level 0 dice (0.52)
   - The learned_alpha blending is degrading coarse predictions
   - Consider: use level 0 only, or fix alpha initialization/training

2. SAMPLING GUIDANCE FAILURE:


In [37]:
print("""
================================================================================
PROPOSED VALIDATION CONFIG CHANGES
================================================================================

PROBLEM ANALYSIS:
1. Model predictions (refined_probs) have ~0.02-0.13 dice → poor sampling guidance
2. Cascade degrades performance: Level 0 (0.52) > Final (0.35)  
3. Current coverage is sparse: 20 patches × 64px = 1280px out of 16384px (7.8%)

================================================================================
OPTION A: SLIDING WINDOW + DENSE COVERAGE (Recommended)
================================================================================
Switch to deterministic sliding_window sampling with full coverage at validation.
This eliminates the failing entropy-based sampling.

levels:
  - resolution: 20
    patch_size: 8
    sampling_method: "sliding_window"  # ← Change from continuous
    stride: 4                          # ← Denser: 5×5=25 patches (was 9)
    num_patches: 9                     # Training (unchanged)
    num_patches_val: 25                # ← Explicit validation count
    num_context_patches: 4
    pad_before: 1
    pad_after: 1
    
  - resolution: 60
    patch_size: 8
    sampling_method: "sliding_window"  # ← Change from continuous  
    stride: 4                          # ← Denser: 14×14=196 patches (was 12)
    num_patches: 12                    # Training (unchanged)
    num_patches_val: 196               # ← Full coverage
    num_context_patches: 6             # ← More context
    pad_before: 1
    pad_after: 1
    
  - resolution: 128
    patch_size: 8
    sampling_method: "sliding_window"  # ← Change from continuous
    stride: 4                          # ← Denser: 31×31=961 patches (was 20)
    num_patches: 20                    # Training (unchanged)
    num_patches_val: 400               # ← ~25% of 961 for memory
    num_context_patches: 8             # ← More context
    pad_before: 0
    pad_after: 0

# Disable cascade blending since it hurts (Level 0 alone beats Final)
cascade:
  confidence_blend: false              # ← Disable confidence blending
  learned_alpha: false                 # ← Disable learned alpha for validation
  
# OR: Force alpha to favor coarser levels
# alpha_head bias init to -2.0 → sigmoid(-2) ≈ 0.12 (favor previous level)
""")



PROPOSED VALIDATION CONFIG CHANGES

PROBLEM ANALYSIS:
1. Model predictions (refined_probs) have ~0.02-0.13 dice → poor sampling guidance
2. Cascade degrades performance: Level 0 (0.52) > Final (0.35)  
3. Current coverage is sparse: 20 patches × 64px = 1280px out of 16384px (7.8%)

OPTION A: SLIDING WINDOW + DENSE COVERAGE (Recommended)
Switch to deterministic sliding_window sampling with full coverage at validation.
This eliminates the failing entropy-based sampling.

levels:
  - resolution: 20
    patch_size: 8
    sampling_method: "sliding_window"  # ← Change from continuous
    stride: 4                          # ← Denser: 5×5=25 patches (was 9)
    num_patches: 9                     # Training (unchanged)
    num_patches_val: 25                # ← Explicit validation count
    num_context_patches: 4
    pad_before: 1
    pad_after: 1

  - resolution: 60
    patch_size: 8
    sampling_method: "sliding_window"  # ← Change from continuous  
    stride: 4                          # 

In [38]:
print("""
================================================================================
OPTION B: SINGLE LEVEL ONLY (Simplest fix)
================================================================================
Since Level 0 alone (0.52) beats the cascade (0.35), just use Level 0 for validation.

# In validation code or eval script:
# Skip levels 1 and 2, return level_0 prediction directly
# final_pred = F.interpolate(level_outputs[0]['pred'], size=(H, W), ...)

# Or in config, set levels 1 & 2 to identity pass-through

================================================================================  
OPTION C: ORACLE VALIDATION (for debugging/ceiling)
================================================================================
Use GT guidance during validation to measure ceiling performance.
This tells you how much the sampling strategy matters.

oracle_levels_valid: [true, true, true]  # ← All oracle
# Expected: dice should improve significantly if sampling is the bottleneck

================================================================================
OPTION D: HYBRID - COARSE ORACLE + FINE MODEL
================================================================================
Use oracle at coarse level (where it works), model predictions at fine level.

oracle_levels_valid: [true, false, false]  
# Level 0: oracle sampling (works well)
# Level 1-2: model-guided (test if better predictions help)

================================================================================
COVERAGE CALCULATIONS (stride analysis)
================================================================================
""")

# Calculate coverage for different strides
import math

def calc_coverage(resolution, patch_size, stride, num_patches=None):
    """Calculate coverage and patch count for sliding window."""
    grid_h = (resolution - patch_size) // stride + 1
    grid_w = (resolution - patch_size) // stride + 1
    total_patches = grid_h * grid_w
    
    # Effective coverage (accounting for overlap)
    covered_pixels = min(resolution * resolution, 
                        grid_h * stride * grid_w * stride + 
                        (patch_size - stride) * (grid_h + grid_w - 1) * stride)
    coverage_pct = (total_patches * patch_size * patch_size) / (resolution * resolution)
    
    if num_patches:
        # For continuous sampling with num_patches
        actual_coverage = (num_patches * patch_size * patch_size) / (resolution * resolution)
        return total_patches, coverage_pct, actual_coverage
    
    return total_patches, coverage_pct, None

print("Level 0 (20px, patch=8):")
for stride in [2, 3, 4, 5, 6, 7]:
    patches, cov, _ = calc_coverage(20, 8, stride)
    print(f"  stride={stride}: {patches:3d} patches, {cov:.1%} coverage")

print("\nLevel 1 (60px, patch=8):")
for stride in [3, 4, 5, 6, 8]:
    patches, cov, _ = calc_coverage(60, 8, stride)
    print(f"  stride={stride}: {patches:3d} patches, {cov:.1%} coverage")

print("\nLevel 2 (128px, patch=8):")
for stride in [4, 6, 8, 10, 12]:
    patches, cov, _ = calc_coverage(128, 8, stride)
    print(f"  stride={stride}: {patches:3d} patches, {cov:.1%} coverage")

print("\nCurrent config (continuous, entropy sampling):")
for level, (res, patches) in enumerate([(20, 9), (60, 12), (128, 20)]):
    _, _, cov = calc_coverage(res, 8, 1, patches)
    print(f"  Level {level} ({res}px): {patches} patches, {cov:.1%} raw coverage")



OPTION B: SINGLE LEVEL ONLY (Simplest fix)
Since Level 0 alone (0.52) beats the cascade (0.35), just use Level 0 for validation.

# In validation code or eval script:
# Skip levels 1 and 2, return level_0 prediction directly
# final_pred = F.interpolate(level_outputs[0]['pred'], size=(H, W), ...)

# Or in config, set levels 1 & 2 to identity pass-through

OPTION C: ORACLE VALIDATION (for debugging/ceiling)
Use GT guidance during validation to measure ceiling performance.
This tells you how much the sampling strategy matters.

oracle_levels_valid: [true, true, true]  # ← All oracle
# Expected: dice should improve significantly if sampling is the bottleneck

OPTION D: HYBRID - COARSE ORACLE + FINE MODEL
Use oracle at coarse level (where it works), model predictions at fine level.

oracle_levels_valid: [true, false, false]  
# Level 0: oracle sampling (works well)
# Level 1-2: model-guided (test if better predictions help)

COVERAGE CALCULATIONS (stride analysis)

Level 0 (20px, patch=8)

In [39]:
print("""
================================================================================
FINAL RECOMMENDATION
================================================================================

The coverage analysis reveals the core issue:
- Level 2 (128px): only 7.8% coverage with 20 patches
- Level 1 (60px): only 21.3% coverage with 12 patches  
- Level 0 (20px): 144% coverage with 9 patches (good overlap)

This explains why Level 0 (0.52 dice) >> Level 2 (0.33 dice)!

================================================================================
RECOMMENDED VALIDATION CONFIG (110_learned_confidence_val.yaml)
================================================================================
""")

config = """
# @package _global_
# Validation-optimized config for 110_learned_confidence
# Changes: sliding_window, denser coverage, disabled cascade blending

defaults:
  - /augmentation
  - override /method: patch_icl

preprocessing:
  image_size: 128

model:
  patch_icl:
    feature_extractor:
      output_grid_size: 128
      freeze: true
      
    levels:
      - resolution: 20
        patch_size: 8
        sampling_method: "sliding_window"   # ← Deterministic
        stride: 4                           # ← 16 patches, 256% coverage
        num_patches: 9
        num_context_patches: 4
        pad_before: 1
        pad_after: 1
        
      - resolution: 60
        patch_size: 8
        sampling_method: "sliding_window"   # ← Deterministic
        stride: 5                           # ← 121 patches, 215% coverage
        num_patches: 12
        num_context_patches: 6              # ← More context
        pad_before: 1
        pad_after: 1
        
      - resolution: 128
        patch_size: 8
        sampling_method: "sliding_window"   # ← Deterministic
        stride: 6                           # ← 441 patches, 172% coverage
        num_patches: 20
        num_context_patches: 8              # ← More context
        pad_before: 0
        pad_after: 0

    oracle_levels_train: [false, true, true]
    oracle_levels_valid: [false, false, false]

    # Keep training settings unchanged
    oracle_scheduling:
      enabled: true
      schedule: "linear"
      start_prob: 1.0
      end_prob: 0.2
      warmup_epochs: 10
      decay_epochs: 30

    sampler:
      target_sampling: "entropy"
      context_sampling: "entropy"

    aggregator:
      type: "average"
      min_coverage: 1e-6
      use_confidence: true
      confidence_mode: "multiply"
      detach_confidence: true

    backbone:
      # ... (unchanged from training config)
      predict_confidence: true

    cascade:
      detach_between_levels: true
      confidence_blend: false              # ← DISABLED for validation
      learned_alpha: false                 # ← DISABLED for validation  
      conf_weighted_aggreg: false
      cascade_registers: true

    loss:
      # ... (unchanged)
"""
print(config)



FINAL RECOMMENDATION

The coverage analysis reveals the core issue:
- Level 2 (128px): only 7.8% coverage with 20 patches
- Level 1 (60px): only 21.3% coverage with 12 patches  
- Level 0 (20px): 144% coverage with 9 patches (good overlap)

This explains why Level 0 (0.52 dice) >> Level 2 (0.33 dice)!

RECOMMENDED VALIDATION CONFIG (110_learned_confidence_val.yaml)


# @package _global_
# Validation-optimized config for 110_learned_confidence
# Changes: sliding_window, denser coverage, disabled cascade blending

defaults:
  - /augmentation
  - override /method: patch_icl

preprocessing:
  image_size: 128

model:
  patch_icl:
    feature_extractor:
      output_grid_size: 128
      freeze: true

    levels:
      - resolution: 20
        patch_size: 8
        sampling_method: "sliding_window"   # ← Deterministic
        stride: 4                           # ← 16 patches, 256% coverage
        num_patches: 9
        num_context_patches: 4
        pad_before: 1
        pad_after: 1

    

In [40]:
print("""
================================================================================
SUMMARY OF KEY CHANGES
================================================================================

| Setting                | Current          | Proposed         | Rationale
|------------------------|------------------|------------------|---------------------------
| sampling_method        | continuous       | sliding_window   | Deterministic, full coverage
| Level 0 stride         | 7 (9 patches)    | 4 (16 patches)   | 144% → 256% coverage
| Level 1 stride         | 6 (12 patches)   | 5 (121 patches)  | 21% → 215% coverage  
| Level 2 stride         | 6 (20 patches)   | 6 (441 patches)  | 8% → 172% coverage
| Level 1 context        | 4                | 6                | More context guidance
| Level 2 context        | 4                | 8                | More context guidance
| confidence_blend       | true             | false            | Cascade hurts performance
| learned_alpha          | true             | false            | Alpha not calibrated yet

================================================================================
EXPECTED IMPACT
================================================================================

1. COVERAGE FIX (biggest win expected):
   - Level 2: 8% → 172% coverage should dramatically improve fine-level dice
   - Expect Level 2 dice to approach Level 1 quality (0.33 → ~0.45)

2. SLIDING WINDOW (deterministic):
   - Eliminates variance from entropy-based sampling
   - No dependence on model predictions (which have poor refined_probs_dice)
   
3. DISABLE CASCADE BLENDING:
   - Level 0 alone (0.52) > Final combined (0.35)
   - Disabling learned_alpha lets levels contribute independently
   - Simple averaging should outperform broken confidence weighting

4. MORE CONTEXT PATCHES:
   - Context provides +0.41 dice benefit (0.57 vs 0.16 local)
   - More context patches = better in-context learning signal

================================================================================
MEMORY CONSIDERATIONS  
================================================================================

441 patches at Level 2 is 22× more than current 20 patches.
If memory is an issue, use stride=8 for 256 patches (13× increase, 100% coverage).

Alternative memory-efficient config:
  - Level 2: stride=8 → 256 patches (100% coverage)
  - Level 1: stride=6 → 81 patches (144% coverage)
""")



SUMMARY OF KEY CHANGES

| Setting                | Current          | Proposed         | Rationale
|------------------------|------------------|------------------|---------------------------
| sampling_method        | continuous       | sliding_window   | Deterministic, full coverage
| Level 0 stride         | 7 (9 patches)    | 4 (16 patches)   | 144% → 256% coverage
| Level 1 stride         | 6 (12 patches)   | 5 (121 patches)  | 21% → 215% coverage  
| Level 2 stride         | 6 (20 patches)   | 6 (441 patches)  | 8% → 172% coverage
| Level 1 context        | 4                | 6                | More context guidance
| Level 2 context        | 4                | 8                | More context guidance
| confidence_blend       | true             | false            | Cascade hurts performance
| learned_alpha          | true             | false            | Alpha not calibrated yet

EXPECTED IMPACT

1. COVERAGE FIX (biggest win expected):
   - Level 2: 8% → 172% coverage should dram